# Comparing Initial and Final Pulsar Distributions

In [ ]:
import numpy as np
import pandas as pd
import pathlib
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from mlpoppyns.simulator.config_simulator import cfg
import mlpoppyns.simulator.basics.constants as const
import utilities.plot_settings

## Loading the data

Import the initial population file.

In [ ]:
path_to_simulation = pathlib.Path("../../data/example_simulation_full_edm/")

data_i = pd.read_pickle(
    pathlib.Path().joinpath(path_to_simulation, "initial_population.pkl.gz"),
    compression="gzip",
)
data_i.head()

In [ ]:
age = data_i["age"]["[yr]"].to_numpy()
r_i = data_i["r"]["[kpc]"].to_numpy()
phi_i = data_i["phi"]["[rad]"].to_numpy()
x_i = r_i * np.cos(phi_i)
y_i = r_i * np.sin(phi_i)
z_i = data_i["z"]["[kpc]"].to_numpy()
vk_r = (
    data_i["v_r"]["[kpc yr^-1]"].to_numpy() * const.KPC_TO_KM / const.YR_TO_S
)
v_phi = (
    data_i["v_phi"]["[kpc yr^-1]"].to_numpy() * const.KPC_TO_KM / const.YR_TO_S
)
vk_z = (
    data_i["v_z"]["[kpc yr^-1]"].to_numpy() * const.KPC_TO_KM / const.YR_TO_S
)
v_orb = (
    data_i["v_orb"]["[kpc yr^-1]"].to_numpy() * const.KPC_TO_KM / const.YR_TO_S
)
vk_phi = v_phi - v_orb
B_i = data_i["B"]["[G]"].to_numpy()
chi_i = data_i["chi"]["[rad]"].to_numpy()
P_i = data_i["P"]["[s]"].to_numpy()
P_dot_i = data_i["P_dot"]["[s s^-1]"].to_numpy()

Import the final population file.

In [ ]:
data_f = pd.read_pickle(
    pathlib.Path().joinpath(path_to_simulation, "final_population.pkl.gz"),
    compression="gzip",
)
data_f.head()

data_PMPS = pd.read_pickle(
    pathlib.Path().joinpath(path_to_simulation, "survey_PMPS_results.pkl.gz"),
    compression="gzip",
)
data_PMPS.head()

data_SMPS = pd.read_pickle(
    pathlib.Path().joinpath(path_to_simulation, "survey_SMPS_results.pkl.gz"),
    compression="gzip",
)
data_SMPS.head()

data_HTRU_low_mid = pd.read_pickle(
    pathlib.Path().joinpath(
        path_to_simulation, "survey_HTRU_low_mid_results.pkl.gz"
    ),
    compression="gzip",
)
data_HTRU_low_mid.head()

data_HTRU_high = pd.read_pickle(
    pathlib.Path().joinpath(
        path_to_simulation, "survey_HTRU_high_results.pkl.gz"
    ),
    compression="gzip",
)
data_HTRU_high.head()

In [ ]:
r_f = data_f["r"]["[kpc]"].to_numpy()
phi_f = data_f["phi"]["[rad]"].to_numpy()
x_f = r_f * np.cos(phi_f)
y_f = r_f * np.sin(phi_f)
z_f = data_f["z"]["[kpc]"].to_numpy()
RA_f = data_f["ra"]["[deg]"].to_numpy()
DEC_f = data_f["dec"]["[deg]"].to_numpy()
pm_RA_f = data_f["pm_ra"]["[mas yr^-1]"].to_numpy()
pm_DEC_f = data_f["pm_dec"]["[mas yr^-1]"].to_numpy()
v_r_f = data_f["v_r"]["[km s^-1]"].to_numpy()
v_phi_f = data_f["v_phi"]["[km s^-1]"].to_numpy()
v_z_f = data_f["v_z"]["[km s^-1]"].to_numpy()
dist_f = data_f["dist"]["[kpc]"].to_numpy()
B_f = data_f["B"]["[G]"].to_numpy()
chi_f = data_f["chi"]["[rad]"].to_numpy()
P_f = data_f["P"]["[s]"].to_numpy()
P_dot_f = data_f["P_dot"]["[s s^-1]"].to_numpy()
L_radio_bol = data_f["L_radio_bol"]["[erg s^-1]"].to_numpy()
w_int = data_f["w_int"]["[s]"].to_numpy()
intercepted_radio = data_f["intercepted_radio"][" "].to_numpy(dtype=bool)

In [ ]:
idx_PMPS = data_PMPS["idx"].to_numpy(dtype=int)
w_PMPS = data_PMPS["w_eff"]["[s]"].to_numpy()

idx_SMPS = data_SMPS["idx"].to_numpy(dtype=int)
w_SMPS = data_SMPS["w_eff"]["[s]"].to_numpy()

idx_HTRU_low_mid = data_HTRU_low_mid["idx"].to_numpy(dtype=int)
w_HTRU_low_mid = data_HTRU_low_mid["w_eff"]["[s]"].to_numpy()

idx_HTRU_high = data_HTRU_high["idx"].to_numpy(dtype=int)
w_HTRU_high = data_HTRU_high["w_eff"]["[s]"].to_numpy()

idx_radio_det = set(
    np.concatenate((idx_PMPS, idx_SMPS, idx_HTRU_low_mid, idx_HTRU_high))
)

idx_radio_det = list(idx_radio_det)

## Analyzing corresponding distributions

Plot of the spatial distribution in galactocentric coordinates. We show in the background in cyan the initial distribution of all the stars, in orange the final distribution of all evolved stars, in blue the initial position of stars that have been detected, and in red the final position of those stars that are detected.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))

ax.plot(
    x_i,
    y_i,
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Initial all",
)
ax.plot(
    x_f,
    y_f,
    linestyle="None",
    marker="o",
    color="tab:orange",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Final all",
)
ax.plot(
    x_i[idx_radio_det],
    y_i[idx_radio_det],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1,
    rasterized=True,
    label="Initial detected radio",
)
ax.plot(
    x_f[idx_radio_det],
    y_f[idx_radio_det],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Final detected radio",
)

ax.plot(0.0, 8.3, marker="o", color="tab:orange", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$y$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.legend(frameon=False, loc=3, fontsize=20, markerscale=3)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

ax.plot(
    x_i,
    z_i,
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Initial all",
)
ax.plot(
    x_f,
    z_f,
    linestyle="None",
    marker="o",
    color="tab:orange",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Final all",
)
ax.plot(
    x_i[idx_radio_det],
    z_i[idx_radio_det],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Initial detected radio",
)
ax.plot(
    x_f[idx_radio_det],
    z_f[idx_radio_det],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Final detected radio",
)

ax.plot(0.0, 0.02, marker="o", color="tab:orange", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$z$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-5.0, 5.0)

plt.legend(frameon=False, loc=3, fontsize=20, markerscale=3)

plt.show()

Galactocentric radius distribution.

In [ ]:
r_i = np.sqrt(x_i**2 + y_i**2)
r_f = np.sqrt(x_f**2 + y_f**2)
r_bins = np.linspace(0.0, 30.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    r_i,
    bins=r_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    ls=":",
    alpha=1,
    label="Initial all",
)
ax.hist(
    r_i[idx_radio_det],
    bins=r_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label="Initial detected radio",
)
ax.hist(
    r_f,
    bins=r_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    ls=":",
    alpha=1,
    label="Final all",
)
ax.hist(
    r_f[idx_radio_det],
    bins=r_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label="Final detected radio",
)

plt.xlabel(r"$r$ [kpc]")
plt.ylabel(r"Number of NSs")
plt.xlim(0.0, 30.0)
plt.yscale("log")
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

Galactic height distribution.

In [ ]:
z_bins = np.linspace(0.0, 5.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    np.abs(z_i),
    bins=z_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    ls=":",
    alpha=1,
    label="Initial all",
)
ax.hist(
    np.abs(z_i[idx_radio_det]),
    bins=z_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label="Initial detected radio",
)
ax.hist(
    np.abs(z_f),
    bins=z_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    ls=":",
    alpha=1,
    label="Final all",
)
ax.hist(
    np.abs(z_f[idx_radio_det]),
    bins=z_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label="Final detected radio",
)

plt.xlabel(r"$z$ [kpc]")
plt.ylabel(r"Number of NSs")
plt.xlim(0.0, 5.0)
plt.yscale("log")
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

Total velocity magnitude distribution.

In [ ]:
v_tot_i = np.sqrt(vk_r**2 + (vk_phi + v_orb) ** 2 + vk_z**2)
v_tot_f = np.sqrt(v_r_f**2 + v_phi_f**2 + v_z_f**2)

fig, ax = plt.subplots(figsize=(15, 8))
v_bins = np.linspace(0, 1600.0, 31)

ax.hist(
    v_tot_i,
    bins=v_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    ls=":",
    alpha=1,
    label=r"Initial all",
)
ax.hist(
    v_tot_i[idx_radio_det],
    bins=v_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Initial detected radio",
)
ax.hist(
    v_tot_f,
    bins=v_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    ls=":",
    alpha=1,
    label=r"final all",
)
ax.hist(
    v_tot_f[idx_radio_det],
    bins=v_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Final detected radio",
)

ax.set_xlabel(r"Total velocity magnitude [km s$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
plt.yscale("log")
plt.ylim(0.1, 1.0e5)
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

Spin period distribution.

In [ ]:
P_bins = np.logspace(-6.0, 3.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    P_i,
    bins=P_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    ls=":",
    label="Initial all",
)
ax.hist(
    P_i[idx_radio_det],
    bins=P_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Initial detected radio",
)
ax.hist(
    P_f,
    bins=P_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    ls=":",
    label="Final all",
)
ax.hist(
    P_f[idx_radio_det],
    bins=P_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Final detected radio",
)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Number of NSs")
# plt.xlim(0., 50.0)
plt.ylim(0.1, 2.0e5)
plt.xscale("log")
plt.yscale("log")
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

Magnetic field distribution.

In [ ]:
B_log10_bins = np.linspace(7.0, 17.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    np.log10(B_i),
    bins=B_log10_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    ls=":",
    label="Initial all",
)
ax.hist(
    np.log10(B_i[idx_radio_det]),
    bins=B_log10_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Initial detected radio",
)
ax.hist(
    np.log10(B_f),
    bins=B_log10_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    ls=":",
    label="Final all",
)
ax.hist(
    np.log10(B_f[idx_radio_det]),
    bins=B_log10_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Final detected radio",
)

plt.xlabel(r"log$_{10} B$ [G]")
plt.ylabel(r"Number of NSs")
plt.ylim(0.1, 1.0e5)
plt.yscale("log")
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

Inclination angle distribution.

In [ ]:
chi_bins = np.linspace(0, np.pi / 2, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    chi_i,
    bins=chi_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    ls=":",
    label="Initial all",
)
ax.hist(
    chi_i[idx_radio_det],
    bins=chi_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Initial detected radio",
)
ax.hist(
    chi_f,
    bins=chi_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    ls=":",
    label="Final all",
)
ax.hist(
    chi_f[idx_radio_det],
    bins=chi_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Final detected radio",
)

plt.xlabel(r"$\chi$ [rad]")
plt.ylabel(r"Number of NSs")
plt.xlim(0.0, np.pi / 2)
plt.ylim(0.1, 1.0e5)
plt.yscale("log")
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

## Relationships between independent parameters

Magnetic field vs spin period.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.plot(
    P_i,
    np.log10(B_i),
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Initial all",
)
ax.plot(
    P_f,
    np.log10(B_f),
    linestyle="None",
    marker="o",
    color="tab:orange",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Final all",
)
ax.plot(
    P_i[idx_radio_det],
    np.log10(B_i[idx_radio_det]),
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Initial detected radio",
)
ax.plot(
    P_f[idx_radio_det],
    np.log10(B_f[idx_radio_det]),
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Final detected radio",
)

ax.set_xlabel(r"$P$ [s]")
ax.set_ylabel(r"log$_{10} B$ [G]")
plt.xscale("log")

plt.legend(frameon=False, loc=3, fontsize=20, markerscale=3)

plt.show()

As a reference we show also the $P-\dot{P}$ diagram.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.plot(
    P_i,
    P_dot_i,
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Initial all",
)
ax.plot(
    P_f,
    P_dot_f,
    linestyle="None",
    marker="o",
    color="tab:orange",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Final all",
)
ax.plot(
    P_i[idx_radio_det],
    P_dot_i[idx_radio_det],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Initial detected radio",
)
ax.plot(
    P_f[idx_radio_det],
    P_dot_f[idx_radio_det],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Final detected radio",
)

ax.set_xlabel(r"$P$ [s]")
ax.set_ylabel(r"$\dot{P}$")
plt.xscale("log")
plt.yscale("log")

plt.legend(frameon=False, loc=3, fontsize=20, markerscale=3)

plt.show()

Inclination angle vs period.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.plot(
    P_i,
    chi_i,
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Initial all",
)
ax.plot(
    P_f,
    chi_f,
    linestyle="None",
    marker="o",
    color="tab:orange",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Final all",
)
ax.plot(
    P_i[idx_radio_det],
    chi_i[idx_radio_det],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Initial detected radio",
)
ax.plot(
    P_f[idx_radio_det],
    chi_f[idx_radio_det],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Final detected radio",
)

ax.set_xlabel(r"$P$ [s]")
ax.set_ylabel(r"$\chi$ [rad]")
plt.xscale("log")

plt.legend(frameon=False, loc=3, fontsize=20, markerscale=3)

plt.show()

Inclination angle vs Bfield.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.plot(
    B_i,
    chi_i,
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Initial all",
)
ax.plot(
    B_f,
    chi_f,
    linestyle="None",
    marker="o",
    color="tab:orange",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Final all",
)
ax.plot(
    B_i[idx_radio_det],
    chi_i[idx_radio_det],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Initial detected radio",
)
ax.plot(
    B_f[idx_radio_det],
    chi_f[idx_radio_det],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Final detected radio",
)

ax.set_xlabel(r"$B$ [G]")
ax.set_ylabel(r"$\chi$ [rad]")
plt.xscale("log")

plt.legend(frameon=False, loc=3, fontsize=20, markerscale=3)

plt.show()